In [ ]:
class ECAAttention(nn.Module):
    def __init__(self, channels, gamma=2, b=1):
        super(ECAAttention, self).__init__()
        t = int(abs((torch.log2(torch.tensor([channels], dtype=torch.float32) + b) / gamma).item()))
        k = t if t % 2 else t + 1

        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv1d(1, 1, kernel_size=k, padding=(k - 1) // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        batch_size, channels, height, width = x.size()
        y = self.avg_pool(x).view(batch_size, channels, 1)  # Global average pooling
        y = self.conv(y.transpose(1, 2)).transpose(1, 2)  # 1D convolution along channel dimension
        y = self.sigmoid(y.view(batch_size, channels, 1, 1))  # Apply sigmoid activation
        return x * y.expand_as(x)  # Channel-wise multiplication